In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time

# Load data
df = pd.read_csv('../data/gtzan_features.csv')

# Split features/target
X = df.drop(['label', 'filename'], axis=1, errors='ignore')
y = df['label']

# Metadata output
print(f"Dataset Shape: {df.shape}")
print(f"Features: {X.shape[1]}")

In [ ]:
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Encode labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Stratified split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.20, random_state=42, stratify=y_encoded
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Cast to DataFrame
X_train_scaled = pd.DataFrame(X_train_scaled, columns=X.columns)
X_test_scaled = pd.DataFrame(X_test_scaled, columns=X.columns)

print("Preprocessing complete.")

In [ ]:
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix

# Define models
models = {
    "SVM": SVC(kernel='rbf', C=10, gamma='scale'),
    "RandomForest": RandomForestClassifier(n_estimators=200, max_depth=None, min_samples_split=5, random_state=42),
    "XGBoost": XGBClassifier(n_estimators=200, max_depth=6, learning_rate=0.1, subsample=0.8, random_state=42),
    "LightGBM": LGBMClassifier(n_estimators=200, learning_rate=0.1, num_leaves=31, random_state=42, verbose=-1)
}

# Benchmarking loop
results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred, average='macro')
    results.append({"Model": name, "Accuracy": acc, "Macro F1": f1})
    
    # Plot confusion matrix
    plt.figure(figsize=(8, 6))
    sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt='d', cmap='Blues', 
                xticklabels=le.classes_, yticklabels=le.classes_)
    plt.title(f'Matrix: {name}')
    plt.show()

print(pd.DataFrame(results))

In [ ]:
from sklearn.model_selection import RandomizedSearchCV, learning_curve

# Parameter search
param_grid = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'num_leaves': [20, 31, 40, 60],
    'max_depth': [-1, 10, 20],
    'subsample': [0.8, 0.9, 1.0]
}

lgbm_rs = RandomizedSearchCV(
    LGBMClassifier(random_state=42, verbose=-1), 
    param_distributions=param_grid, 
    cv=3, n_iter=15, scoring='f1_macro', n_jobs=-1, random_state=42
)
lgbm_rs.fit(X_train_scaled, y_train)

# Learning curve
train_sizes, train_scores, test_scores = learning_curve(
    lgbm_rs.best_estimator_, X_train_scaled, y_train, cv=3, scoring='f1_macro', 
    n_jobs=-1, train_sizes=np.linspace(0.1, 1.0, 5)
)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, np.mean(train_scores, axis=1), label="Train")
plt.plot(train_sizes, np.mean(test_scores, axis=1), label="CV")
plt.title("Learning Curve")
plt.legend()
plt.show()

In [ ]:
# Feature importance
best_lgbm = lgbm_rs.best_estimator_
pd.Series(best_lgbm.feature_importances_, index=X.columns).nlargest(20).plot(kind='barh')
plt.show()

# Error analysis
y_pred_final = best_lgbm.predict(X_test_scaled)
cm = confusion_matrix(y_test, y_pred_final)
confusions = []
for i in range(len(le.classes_)):
    for j in range(len(le.classes_)):
        if i != j and cm[i, j] > 0:
            confusions.append((le.classes_[i], le.classes_[j], cm[i, j]))
confusions.sort(key=lambda x: x[2], reverse=True)
print("Top 3 Confused Pairs:")
for p1, p2, count in confusions[:3]:
    print(f"{p1}/{p2}: {count}")

In [ ]:
import joblib

# Commented out to prevent accidental overwriting during development. Uncomment to export models.

# Export components
# joblib.dump(best_lgbm, '../models/genre_classifier.pkl')
# joblib.dump(scaler, '../models/genre_scaler.pkl')
# joblib.dump(le, '../models/genre_encoder.pkl')

print("Export complete.")